In [ ]:
#***************************************************************************************************#
# CODE DEVELOPED BY CHISOMO DAKA - MARCH, 2025 (WITS UNIVERSITY - NANO-SCALE TRANSPORT PHYSICS LAB) #
#***************************************************************************************************#
import json
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import ZFeatureMap, ZZFeatureMap
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import BaseEstimatorV2, StatevectorEstimator as BaseEstimatorV2, Estimator
from qiskit.primitives import BaseSamplerV2, BaseSamplerV1, StatevectorSampler as BaseSamplerV2, BaseSamplerV1, StatevectorSampler
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import Optimize1qGates, Collect2qBlocks, UnitarySynthesis
from qiskit_machine_learning.optimizers import COBYLA, POWELL, NELDER_MEAD #Gradient Free Optimizers
from qiskit_machine_learning.utils import algorithm_globals
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier, VQC
from qiskit_machine_learning.neural_networks import EstimatorQNN, SamplerQNN
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA #Use PCA -> Principal Component Analysis to further reduce feature dimensions (Optional)
#*****************************
algorithm_globals.random_seed = 12345 #Use 12345 as the starting point for all random number generation across algorithms, circuits, and simulators that depend on randomness.

#*********************
#GENERALIZED CONVOLUTIONAL AND POOLING UNITARIES
# We now define a generalized two-qubit convolutional unitary
num_convolutional_params = 2 #params
def conv_circuit(params):
    target = QuantumCircuit(2)
    target.rz(np.pi / 2, 1)
    target.cx(1, 0)
    target.rz((2*params[0]-np.pi/2), 0)
    target.ry((np.pi/2-2*params[1]), 1)
    target.cx(1, 0)
    target.rz(-np.pi / 2, 0)
    return target

# We now define a generalized two-qubit pooling unitary
num_pooling_params = 2 #params
def pool_circuit(params):
    target = QuantumCircuit(2)
    target.rz(-np.pi / 2, 1)
    target.cx(1, 0)
    target.rz((2*params[0]-np.pi/2), 0)
    target.ry((np.pi/2-2*params[1]), 1)
    target.cx(0, 1)
    target.ry(np.pi / 2, 0)
    return target

#GENERALIZED CONVOLUTIONAL LAYER AND POOLING LAYER (with Sources & Sinks)
# We now define a generalized Q-qubit convolutional layer: Q = num_qubits -> even
def conv_layer(num_qubits, param_prefix, iterX):
    qc = QuantumCircuit(num_qubits, name="Convolutional Layer-"+str(iterX))
    qubits = list(range(num_qubits))
    param_index = 0
    params = ParameterVector(param_prefix, length=num_qubits * num_convolutional_params)
    for q1, q2 in zip(qubits[0::2], qubits[1::2]):
        qc = qc.compose(conv_circuit(params[param_index : (param_index + num_convolutional_params)]), [q1, q2])
        qc.barrier()
        param_index += num_convolutional_params
    for q1, q2 in zip(qubits[1::2], qubits[2::2] + [0]):
        qc = qc.compose(conv_circuit(params[param_index : (param_index + num_convolutional_params)]), [q1, q2])
        qc.barrier()
        param_index += num_convolutional_params

    qc_inst = qc.to_instruction()

    qc = QuantumCircuit(num_qubits)
    qc.append(qc_inst, qubits)
    #qc.barrier() # Add barrier after QCNN convolutional layer
    print("The convolutional layer circuit below has", param_index, "parameters")
    return qc

# We now define a generalized Q-qubit pooling layer: Q = num_qubits
def pool_layer(sources, sinks, param_prefix, iterX):
    #num_qubits = num_qbts
    num_qubits = len(sources) + len(sinks)
    print("Pooling qubits:", num_qubits)
    qc = QuantumCircuit(num_qubits, name="Pooling Layer-"+str(iterX))
    param_index = 0
    params = ParameterVector(param_prefix, length=num_qubits // 2 * num_pooling_params)
    for source, sink in zip(sources, sinks):
        qc = qc.compose(pool_circuit(params[param_index : (param_index + num_pooling_params)]), [source, sink])
        qc.barrier()
        param_index += num_pooling_params

    qc_inst = qc.to_instruction()

    qc = QuantumCircuit(num_qubits)
    qc.append(qc_inst, range(num_qubits))
    #qc.barrier()
    return qc

#Create an auto generation function for sources and sinks lists -> pooling layer outputs
def generate_sources_sinks (num_qubits):
    #declare empty sources and sinks lists
    sources, sinks = [],[]
    #Proceed only if the number of qubits is even
    if num_qubits%2 == 0:
        for i in range(num_qubits):
            if i < int(num_qubits/2): 
                sources.append(i) # sources are the first half number of qubits
            else:                     
                sinks.append(i)   # sinks are the last half number of qubits
    #Report error for old number of qubits
    else:
        print("You have provided an old number of qubits!!!")
        #return sources and sinks
    return sources, sinks

#GENARATING BINARY IMAGE DATASET (N = num_images)
#Generate 2^n x 2^n pixelated image dataset with 2^n-pixel horizontal and 2^n-pixel vertical line attributes (binary)
def generate_binary_dataset(num_images, num_qubits):
    images = [] #Declare Images Empty List
    labels = [] #Declare Image Lables Empty List
    #Horizontal Lines
    hor_array = np.zeros((int(np.sqrt(num_qubits)), num_qubits)) # 2^n rows and 2^n columns filled with zeros (8 possible 8 pixel horizontal image lines, each line on a 64 pixel grid)
    #hor_line_start_points = [0, 8, 16, 24, 32, 40, 48, 56] # 8 starting points for 8 pixel horizontal lines
    hor_line_start_points = []
    for i in range(int(np.sqrt(num_qubits))):
         hor_line_start_points.append(i*int(np.sqrt(num_qubits)))
    #Vertical Lines
    ver_array = np.zeros((int(np.sqrt(num_qubits)), num_qubits)) # 8 rows and 64 columns filled with zeros (8 possible 8 pixel veritical image lines, each line on a 64 pixel grid)
    #ver_line_start_points = [0, 1, 2, 3, 4, 5, 6, 7] # 8 starting points for 8 pixel vertical lines
    ver_line_start_points =[]
    for j in range(int(np.sqrt(num_qubits))):
         ver_line_start_points.append(j)
    #Populating n Pissible Horizontal Lines
    for i in hor_line_start_points: #8 possible 8 pixel horizontal image lines, each line on a 64 pixel grid
            #print (i)
            for x in range(0, int(np.sqrt(num_qubits))): # Every horizontal line has 8 sequential pixels (runs from left to right)
                hor_array[int(i/int(np.sqrt(num_qubits)))][i+x] = np.pi / 2 # (hor_array[row#][column#]) polpulating 8 sequential pixels in each line (x E [0, 7])
    #print (hor_array) # print horizontal lines

    #Populating n Pissible Vertical Lines
    for j in ver_line_start_points: #8 possible 8 pixel vertical image lines, each line on a 64 pixel grid
            #print (j)
            for y in range(0, int(np.sqrt(num_qubits))): # Every vertical line has 8 sequential pixels (runs from top to bottom)
                ver_array[j][j+(y*int(np.sqrt(num_qubits)))] = np.pi / 2 # (ver_array[row#][column#]) polpulating 8 sequential pixels in each vertical line (y E [0, 7])
    #print (ver_array) # print vertical lines

    #Generate "num_images" nxn Random Line Images from Horizontal and Vertical Lines
    #Balance the Dataset (alternatively) by ensuring the same number of samples and variations in each class
    x, y = 0, 0 # starting line image positions
    for n in range(num_images): # n images
        #rng = algorithm_globals.random.integers(0, 2) # select 0 or 1 randomly (~tossing a coin)
        #if rng == 0: # if zero (0) is selected, generate a random horizontal line
        if n % 2 == 0: # First half dataset -> Horizontal Line Images
            labels.append(-1) # append -1 label for horizontal line
            #random_image = algorithm_globals.random.integers(0, int(np.sqrt(num_qubits))) # select a random horizontal line among the 8 possible lines
            #images.append(np.array(hor_array[random_image])) # append the randomly selected horizontal line to "image date set"
            images.append(np.array(hor_array[y]))
            y += 1
            if y >= int(np.sqrt(num_qubits)):
                 y = 0
        #elif rng == 1: # if one (1) is selected, generate a random verticla line
        else: # Last half dataset -> Horizontal Line Images
            labels.append(+1) # append +1 label for vertical line
            #random_image = algorithm_globals.random.integers(0, int(np.sqrt(num_qubits))) # select a random vertical line among the 8 possible lines
            #images.append(np.array(ver_array[random_image])) # append the randomly selected vertical line to "image date set"
            images.append(np.array(ver_array[x]))
            x += 1
            if x >= int(np.sqrt(num_qubits)):
                 x = 0

        # Insert Backgound noise for each randomly created image line (noise level E [0, pi/4])
        # This adds noise, preventing the model from memorizing "line patterns" perfectly.
        for i in range(num_qubits): # for each pixel "i" among the 64 pixels in an 8x8 pixel image line...
            if images[-1][i] == 0: # choose pixels with zero values (to avoid perfect pattern memory)
                images[-1][i] = algorithm_globals.random.uniform(0, np.pi / 4) # create background noise if the "i-th" pixel is not an image data pixel (i.e. i=0)
                
    return images, labels #return image data with labels

#INITIALIZE EXPERIMENT VARIABLES
n = 2 #4x4 Imagescorresponds to number of conv-pool layers -> l = n
num_conv_pool_layers = 2*n-2 # num conv-pool-layers (based on image square size -> n)
num_initialized_qubits_Q = 2**n * 2**n #(Q = 2^n x 2^n)
num_binary_image_dataset =100 # size of binary dataset N
# TR% -> Training Sample Size and TE% -> Testing Sample Size # Vary training-testing dataset ratio: 70:30, 80:20, or 90:10
train_data_size = 0.7 # 70% training data
test_data_size = 0.3 #30% test data

#GENERATE FULL BINARY IMAGE DATASET (WITH TRAINING AND TESTING SUB-DATASETS) -> Size N
# Generate a pixelated binary dataset with N-image samples
images, labels = generate_binary_dataset(num_binary_image_dataset, num_initialized_qubits_Q)
if (len(images) == num_binary_image_dataset and len(labels) == num_binary_image_dataset):
    print("A well balanced Block Matrix Image Dataset with", num_binary_image_dataset, " images has been created successfully")

# Generate training and testing sub-datasets (images and labels)
train_images, test_images, train_labels, test_labels = train_test_split(
    images,
    labels, 
    train_size=train_data_size,
    test_size=test_data_size, 
    random_state=246, 
    stratify=labels #Ensures that both train_labels and test_labels have the same class distribution as original dataset labels
)

#BUILD THE No-QCNN NEURAL NETWORK FRAMEWORK
#Declare estimator & base estimator v2 (optional -> Qiskit Primitives)
estimatorV1 = Estimator()
estimatorV2 = BaseEstimatorV2()
SV_sampler = StatevectorSampler()
#samplerV1 = BaseSamplerV1()
samplerV2 = BaseSamplerV2() #BaseSamplerV2, BaseSamplerV1, StatevectorSampler
base_pass_manager = PassManager([Collect2qBlocks(),UnitarySynthesis()])
#z_feature_map = ZFeatureMap(num_initialized_qubits_Q) #No-QCNN data Encoding feature map -> pixel wise encoding
#Declare the ZZFeatureMap -> Linear Entanglement
zz_feature_map = ZZFeatureMap(num_initialized_qubits_Q, reps=1, insert_barriers=True, entanglement="linear") #No-QCNN data Encoding feature map -> pixel wise encoding

#Generating l = n Convolutional-Pooling Layers (num_qubits should be powers of 2 -> special even numbers -> n = 2++)
def generate_conv_pool_layers(num_qbts, num_layers_l):
    #Declare No-QCNN Convolutional-Pooling Circuit Layers
    conv_pool_qc_layers = QuantumCircuit(num_qbts, name="Conv_Pool_Circuit_Layers")
    # number of layers l = n
    #n = int(np.log2(num_qbts)) ->
    No_QCNN_layers = num_layers_l# N = n (i.e. num layers l = n, for num_qubits Q = 2^n x 2^n)
    for i in range(0, No_QCNN_layers):
        if i == 0:
            start_point = 0
            remainder = 0
        else:
            remainder = num_qbts - start_point
            start_point = start_point + int (remainder/2)

        print("Start point:", start_point)
        # n-th Convolutional Layer
        conv_pool_qc_layers.compose(conv_layer(int(num_qbts/(2**i)), "c"+str(i+1), i+1), list(range(start_point, num_qbts)), inplace=True, inline_captures=False)
        # n-th Pooling Layer
        #sources, sinks = generate_sources_sinks (int(num_qbts/(2**i)))
        print('Sources + Sinks :', len(list(range(0,int(num_qbts/(2**(i+1)))))) + len(list(range(int(num_qbts/(2**(i+1))), int(num_qbts/((2**(i))))))))
        conv_pool_qc_layers.compose(pool_layer(list(range(0,int(num_qbts/(2**(i+1))))), list(range(int(num_qbts/(2**(i+1))), int(num_qbts/((2**(i)))))), "p"+str(i+1), i+1), list(range(start_point, num_qbts)), inplace=True)
        #barrier after each (conv+pool) layer
        conv_pool_qc_layers.barrier()
    #Return Total Successive No-QCNN Convolutional-Pooling Circuit Layers
    return conv_pool_qc_layers

#Assign No-QCNN Convolutional-Pooling Circuit Layers (with l = n)
No_QCNN_conv_pool_layers = generate_conv_pool_layers(num_initialized_qubits_Q, num_conv_pool_layers)

#Generate No-QCNN Feature Map -> Combining the ZZFeatureMap and No-QCNN Convolutional-Pooling Circuit Layers
def generate_feature_map_observable(num_qbts, conv_pool_layers, FM):
    #Declare No-QCNN Feature Map Circuit
    FM_circuit = QuantumCircuit(num_qbts)
    FM_circuit.compose(FM, range(num_qbts), inplace=True)
    FM_circuit.barrier() #barrier after feature map (for visual display only)
    FM_circuit.compose(conv_pool_layers, range(num_qbts), inplace=True)
    #Declare No-QCNN Observable
    FM_observable = SparsePauliOp.from_list([("I" * (num_qbts-4) + "ZZZZ", 1)]) # No-QCNN Observable in Z-basis -> Fully connected circuit on last 4 Q-bits
    #Return No-QCNN Feature Map and Observable
    return FM_circuit, FM_observable

#Initialize No-QCNN Feature Map and Observable (num_qubits, conv-pool-layers, zzfeaturemap)
No_QCNN_Feature_Map, No_QCNN_Observable = generate_feature_map_observable(num_initialized_qubits_Q, No_QCNN_conv_pool_layers, zz_feature_map)

#Binary No-QCNN Neural Network
def generate_binary_No_QCNN_neural_network(No_QCNN_FM, No_QCNN_obs, zz_FM, conv_pool_layers):
    binary_neural_network = EstimatorQNN(
        circuit=No_QCNN_FM.decompose(), # we decompose the No-QCNN Feature Map circuit to avoid additional data copying
        pass_manager = None, #base_pass_manager #optional pass_manager instance for primitives that require transpilation (cloud)
        observables=No_QCNN_obs, # defines the expectation value computation basis
        input_params=zz_FM.parameters, # list of quantum circuit parameters that should be treated as “network inputs”
        weight_params=conv_pool_layers.parameters, # list of quantum circuit parameters that should be treated as “network weights”
        estimator=estimatorV2, #optional primitive instance
    )
    return binary_neural_network

No_QCNN_neural_network_sampler=SamplerQNN(
    circuit=No_QCNN_Feature_Map.decompose(),
    input_params=zz_feature_map.parameters, # list of quantum circuit parameters that should be treated as “network inputs”
    weight_params=No_QCNN_conv_pool_layers.parameters, # list of quantum circuit parameters that should be treated as “network weights”
    sampler=SV_sampler
)

#Initialize the Fully Composed No-QCNN Binary Neural Network Framework
No_QCNN_binary_neural_network = generate_binary_No_QCNN_neural_network(No_QCNN_Feature_Map,No_QCNN_Observable,zz_feature_map,No_QCNN_conv_pool_layers)

#SETUP THE BINARY CLASSIFICATION EXPERIMENT
#Intitialize Variables
weights_vals = []
objective_func_vals = []
max_opt_iter = 100 #Maximum ieterations
epoch_list = [5, 10, 15, 20] #list of training epochs

#Training Image Dataset and Labels Arrays
train_images_array = np.asarray(train_images)
train_labels_array = np.asarray(train_labels)

#Set Initial Evaluation Point -> If known from prior experiment
initial_point = None

#Call Back Objective Function
def callback_graphs(weights, obj_func_eval):
    #Customize Callback for Advanced Plotting etc.
    clear_output(wait=True)
    weights_vals.append(weights) 
    objective_func_vals.append(obj_func_eval)
    plt.title("Objective Function Vs Iteration")
    plt.xlabel("Iteration")
    plt.ylabel("Objective function value")
    plt.plot(range(len(objective_func_vals)), objective_func_vals)
    plt.show()

#Initialize VQC Classifier with COBYLA (Linear Optimizer) - Cross Entropy Loss
No_QCNN_vqc_classifier = VQC(
    num_qubits=num_initialized_qubits_Q,
    feature_map=zz_feature_map,
    ansatz=No_QCNN_conv_pool_layers,
    loss="cross_entropy",
    optimizer=COBYLA(maxiter=max_opt_iter),  # Set max iterations here -> COBYLA
    output_shape= 2, #Binary Classification
    callback=callback_graphs,
    sampler=SV_sampler, #SV_sampler or SamplerV2,
    warm_start= False,
    initial_point= initial_point #initial_point -> None
)

plt.rcParams["figure.figsize"] = (12, 6)

No_QCNN_vqc_classifier.fit(train_images_array, train_labels_array) #Training No-QCNN

# score No-QCNN VQC Classifier
print(f"Accuracy from the train data : {np.round(100 * No_QCNN_vqc_classifier.score(train_images_array, train_labels_array), 2)}%")